In [1]:
import pandas as pd
import numpy as np
import requests
from requests.exceptions import HTTPError
import os
from dotenv import load_dotenv
from etl import extract
import datetime

load_dotenv()
access_key = os.getenv('ACCESS_KEY')

# Define API endpoint
url = 'https://www.goflightlabs.com/flights'

# Extracting data from API endpoint
flight_data_raw = extract(url, access_key)
print(flight_data_raw.shape)

Returned status code: 200
Returned status code: 200
(100, 23)


In [2]:
# EDA
display(flight_data_raw.head(), flight_data_raw.info(), flight_data_raw.describe())

print('\nNumber of null values for every column feature\n')
flight_data_raw.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   hex            100 non-null    object 
 1   reg_number     100 non-null    object 
 2   flag           100 non-null    object 
 3   lat            100 non-null    float64
 4   lng            100 non-null    float64
 5   alt            100 non-null    int64  
 6   dir            100 non-null    float64
 7   speed          100 non-null    int64  
 8   v_speed        100 non-null    int64  
 9   flight_number  100 non-null    object 
 10  flight_icao    100 non-null    object 
 11  flight_iata    100 non-null    object 
 12  dep_icao       100 non-null    object 
 13  dep_iata       100 non-null    object 
 14  arr_icao       100 non-null    object 
 15  arr_iata       100 non-null    object 
 16  airline_icao   100 non-null    object 
 17  airline_iata   100 non-null    object 
 18  aircraft_ic

,hex,reg_number,flag,lat,lng,alt,dir,speed,v_speed,flight_number,...,dep_iata,arr_icao,arr_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type,squawk
0,46B8B1,SX-NEQ,GR,41.579222,18.984172,11960,323.0,775,0,604,...,ATH,EGLL,LHR,AEE,A3,A20N,1761837121,en-route,adsb,NaN
1,407EEB,G-SUNF,UK,53.127981,0.029860,10398,112.0,906,0,943,...,MAN,LCLK,LCA,EXS,LS,A21N,1761837121,en-route,adsb,NaN
2,A6BFA6,N534DT,US,34.259226,-94.111978,10055,270.7,746,0,777,...,ATL,KLAS,LAS,DAL,DL,A21N,1761837121,en-route,adsb,NaN
3,A0B7B6,N14502,US,41.059485,-96.665857,10474,82.0,982,0,1996,...,ORD,TJSJ,SJU,UAL,UA,A21N,1761837121,en-route,adsb,NaN
4,AA6E58,N771SK,US,42.093091,-109.704807,11244,311.3,733,0,4788,...,DEN,KIDA,IDA,SKW,OO,CRJ7,1761837121,en-route,adsb,NaN


None

,lat,lng,alt,dir,speed,v_speed,updated
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.0,1.000000e+02
mean,21.212607,12.109404,9418.740000,185.917000,778.280000,0.0,1.761837e+09
std,27.778804,75.068410,3081.430314,95.105228,152.984853,0.0,1.714466e-01
min,-39.781962,-141.167895,269.000000,5.100000,251.000000,0.0,1.761837e+09
25%,3.895743,-49.447680,9300.500000,113.275000,746.750000,0.0,1.761837e+09
50%,25.637226,14.652945,10653.500000,191.900000,809.500000,0.0,1.761837e+09
75%,43.028603,79.193750,11302.750000,271.175000,871.000000,0.0,1.761837e+09
max,60.119520,142.976585,12547.000000,358.600000,1000.000000,0.0,1.761837e+09



Number of null values for every column feature



hex               0
reg_number        0
flag              0
lat               0
lng               0
alt               0
dir               0
speed             0
v_speed           0
flight_number     0
flight_icao       0
flight_iata       0
dep_icao          0
dep_iata          0
arr_icao          0
arr_iata          0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
squawk           96
dtype: int64

<h1>Data Cleaning</h1>

<li>Creating a copy of the raw flight data and applying data cleaning</li>
<li>Replacing missing values in "squawk" column with "unknown"</li>
<li>Replacing missing values in 'alt', 'speed' and v_speed to 0</li>
<li>Renaming columns</li>
<li>Converting "updated" values to datetime</li>


In [10]:
def transform(df):
    # Copying raw flight data
    df_copy = df.copy()

    # Replacing missing values with default values
    default_vals = {'squawk': 'Unknown', 'alt': 0, 'speed': 0, 'v_speed': 0.0}
    df_copy = df_copy.fillna(default_vals)

    # Removing duplicates
    df_copy = df_copy.drop_duplicates(subset=['hex', 'updated'])

    # Converting to imperial 
    df_copy['alt'] = df_copy['alt'].apply(lambda x: x * 3.28084)
    df_copy[['speed', 'v_speed']] = df_copy[['speed', 'v_speed']].apply(lambda x: x * 0.621371)


    # Converting 'updated' values to datetime
    df_copy['updated'] = df_copy['updated'].apply(lambda x: datetime.datetime.fromtimestamp(x))

    # Renaming columns
    renamed_columns = {
        'alt': 'altitude_ft', 'speed': 'speed_mph', 'v_speed': 'v_speed_mph', 'lat': 'latitude', 'lng': 'longitude',
        'dep_icao': 'departure_icao', 'dep_iata': 'departure_iata', 'arr_icao': 'arrival_icao', 'arr_iata': 'arrival_iata'
        }
    
    df_copy = df_copy.rename(columns=renamed_columns)

    print('transform complete')

    return df_copy

raw2 = flight_data_raw.copy()
clean = transform(raw2)

display(clean.head())

print('\nNumber of null values for every column feature\n')
clean.isnull().sum()


transform complete


,hex,reg_number,flag,latitude,longitude,altitude_ft,dir,speed_mph,v_speed_mph,flight_number,...,departure_iata,arrival_icao,arrival_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type,squawk
0,46B8B1,SX-NEQ,GR,41.579222,18.984172,39238.84640,323.0,481.562525,0.0,604,...,ATH,EGLL,LHR,AEE,A3,A20N,2025-10-30 11:12:01,en-route,adsb,Unknown
1,407EEB,G-SUNF,UK,53.127981,0.029860,34114.17432,112.0,562.962126,0.0,943,...,MAN,LCLK,LCA,EXS,LS,A21N,2025-10-30 11:12:01,en-route,adsb,Unknown
2,A6BFA6,N534DT,US,34.259226,-94.111978,32988.84620,270.7,463.542766,0.0,777,...,ATL,KLAS,LAS,DAL,DL,A21N,2025-10-30 11:12:01,en-route,adsb,Unknown
3,A0B7B6,N14502,US,41.059485,-96.665857,34363.51816,82.0,610.186322,0.0,1996,...,ORD,TJSJ,SJU,UAL,UA,A21N,2025-10-30 11:12:01,en-route,adsb,Unknown
4,AA6E58,N771SK,US,42.093091,-109.704807,36889.76496,311.3,455.464943,0.0,4788,...,DEN,KIDA,IDA,SKW,OO,CRJ7,2025-10-30 11:12:01,en-route,adsb,Unknown



Number of null values for every column feature



hex               0
reg_number        0
flag              0
latitude          0
longitude         0
altitude_ft       0
dir               0
speed_mph         0
v_speed_mph       0
flight_number     0
flight_icao       0
flight_iata       0
departure_icao    0
departure_iata    0
arrival_icao      0
arrival_iata      0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
squawk            0
dtype: int64